# LangChain Agents with Euron AI

## Setup Instructions

1. **Select the correct kernel**: `Python 3.10 (Agents)` or `agents_py310`
2. **Create `.env` file** with your API key:
   ```
   EURON_API_KEY=your_api_key_here
   ```
3. **Run cells in order**

## Important Notes

- ✅ Uses LangChain 1.x (modern version)
- ✅ API key loaded from `.env` file (secure)
- ✅ Compatible with Pydantic 2.x
- ✅ Uses `InMemoryChatMessageHistory` for conversation memory
- ✅ Uses `langgraph.prebuilt.create_react_agent` for agents

In [ ]:
# Load environment variables
import os
from dotenv import load_dotenv

# Load .env file
load_dotenv()

# Get API key from environment
api_key = os.getenv("EURON_API_KEY")

if not api_key:
    raise ValueError("EURON_API_KEY not found in .env file. Please create .env file with your API key.")

print("✅ Environment loaded successfully")
print(f"✅ API key found: {api_key[:10]}...{api_key[-10:]}")

In [ ]:
# Test basic chat model
from euriai.langchain import create_chat_model

chat_model = create_chat_model(
    api_key=api_key,
    model="gpt-4.1-nano",
    temperature=0.7
)

response = chat_model.invoke("What is artificial intelligence?")
print(response.content)

In [ ]:
# Import required libraries (LangChain 1.x compatible)
import ast, re, math, os, sys
from euriai.langchain import create_chat_model
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda, RunnableWithMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory as ChatMessageHistory
from langchain_core.tools import BaseTool, tool
from langgraph.prebuilt import create_react_agent

print("✅ All imports successful")

## Agent Architecture

We're creating multiple specialized agents:

1. **Calculation Agent** - Mathematical operations
2. **Knowledge Search Agent** - RAG/Vector DB operations
3. **SQL Agent** - Database interactions

### Steps for Creating Agents:

1. Create LLM
2. Define Functions
3. Decorate as Tools
4. Create Instruction Prompt
5. Add Memory
6. Create Agent Executor

In [ ]:
# Define custom tools using @tool decorator

@tool
def calculator(expression: str) -> str:
    """Evaluates a mathematical expression safely.
    
    Args:
        expression: A mathematical expression as a string (e.g., '2 + 2', '10 * 5')
    
    Returns:
        The result of the calculation as a string
    """
    try:
        # Safe evaluation using ast.literal_eval for simple expressions
        result = eval(expression, {"__builtins__": {}}, {"math": math})
        return str(result)
    except Exception as e:
        return f"Error: {str(e)}"

@tool
def string_length(text: str) -> str:
    """Returns the length of a string.
    
    Args:
        text: The input string
    
    Returns:
        The length of the string
    """
    return str(len(text))

@tool
def reverse_string(text: str) -> str:
    """Reverses a string.
    
    Args:
        text: The input string to reverse
    
    Returns:
        The reversed string
    """
    return text[::-1]

# List of tools
tools = [calculator, string_length, reverse_string]

print("✅ Tools created:")
for tool_obj in tools:
    print(f"  - {tool_obj.name}: {tool_obj.description}")

In [ ]:
# Create the agent using LangGraph (LangChain 1.x compatible)

# Create chat model
llm = create_chat_model(
    api_key=api_key,
    model="gpt-4.1-nano",
    temperature=0.7
)

# Create agent with tools
agent_executor = create_react_agent(llm, tools)

print("✅ Agent created successfully")

In [ ]:
# Test the agent

# Test 1: Calculator
print("Test 1: Calculator")
response = agent_executor.invoke({"messages": [("user", "What is 25 * 4 + 10?")]})
print(response["messages"][-1].content)
print()

# Test 2: String operations
print("Test 2: String operations")
response = agent_executor.invoke({"messages": [("user", "What is the length of 'Hello World'?")]})
print(response["messages"][-1].content)
print()

# Test 3: Reverse string
print("Test 3: Reverse string")
response = agent_executor.invoke({"messages": [("user", "Reverse the string 'LangChain'")]})
print(response["messages"][-1].content)

In [ ]:
# Add conversation memory

# Store for conversation histories
store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    """Get or create a chat history for a session."""
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# Create agent with memory
agent_with_memory = RunnableWithMessageHistory(
    agent_executor,
    get_session_history,
    input_messages_key="messages",
    output_messages_key="messages",
)

print("✅ Agent with memory created")

In [ ]:
# Test agent with memory

config = {"configurable": {"session_id": "test-session-1"}}

# First message
print("Message 1:")
response = agent_with_memory.invoke(
    {"messages": [("user", "My name is Alice. Calculate 10 + 5 for me.")]},
    config=config
)
print(response["messages"][-1].content)
print()

# Second message (should remember name)
print("Message 2:")
response = agent_with_memory.invoke(
    {"messages": [("user", "What's my name? And what was the result of the calculation?")]},
    config=config
)
print(response["messages"][-1].content)

## Next Steps

1. **Add more tools**: Create tools for RAG, SQL, web search, etc.
2. **Improve prompts**: Customize system prompts for better agent behavior
3. **Add error handling**: Implement retry logic and error recovery
4. **Create specialized agents**: Build domain-specific agents
5. **Add logging**: Track agent decisions and tool usage

## Resources

- [LangChain Documentation](https://python.langchain.com/)
- [LangGraph Documentation](https://langchain-ai.github.io/langgraph/)
- [Euron AI Documentation](https://euron.one/)